# 04. 발신 통화 분류 검증 + 재분류

`03_analyze_outbound_calls.ipynb`(또는 `scripts/analyze_outbound_calls.py`)가 만든 `data/outbound/outbound_call_checkpoint.json`을 대상으로
분류 품질을 점검합니다.

- **같은 수신번호인데 고객콜백/공급업체가 섞여 분류된 경우 확인** — 03의 분류 프롬프트는 "상대방이 누구인가"를 기준으로 판단하지만, 통화 내용에 따라 여전히 오분류가 있을 수 있습니다.
- **"기타"로 분류된 통화 중 실제로는 전사문이 있는 건을 재검토하고, 필요하면 Ollama로 재분류하여 체크포인트에 반영**

- **입력**: `../data/outbound/outbound_call_checkpoint.json`
- **주의**: 이 노트북의 마지막 섹션(재분류 반영)은 원본 체크포인트를 직접 수정합니다. `APPLY=True`로 실행하기 전 체크포인트를 별도 백업해두는 것을 권장합니다.

In [ ]:
# ============================================================
# 0. 설정 + 체크포인트 로드
# ============================================================
import json
import os

import pandas as pd
import requests

CHECKPOINT_FILE = "../data/outbound/outbound_call_checkpoint.json"

with open(CHECKPOINT_FILE, encoding="utf-8") as f:
    ckpt = json.load(f)

results = ckpt["results"]
df = pd.DataFrame(results)


def extract_receiver(fname):
    parts = str(fname).split("_")
    return parts[1] if len(parts) > 1 else ""


df["receiver"] = df["file_name"].apply(extract_receiver)

print(f"총 {len(df)}건 로드")
print(df["call_type"].value_counts())

---
## 1. 번호별 혼재 분류 확인 (공급업체 ↔ 고객콜백)

In [ ]:
# 번호별 분류 현황
type_by_num = df.groupby("receiver")["call_type"].value_counts().unstack(fill_value=0)
type_by_num["total"] = type_by_num.sum(axis=1)
type_by_num = type_by_num.sort_values("total", ascending=False)
print(f"고유 수신번호: {len(type_by_num)}개")
type_by_num.head(20)

In [ ]:
# 공급업체 + 고객콜백 둘 다 있는 번호 (혼재 번호)
has_supplier = type_by_num.get("공급업체", pd.Series(0, index=type_by_num.index)) > 0
has_callback = type_by_num.get("고객콜백", pd.Series(0, index=type_by_num.index)) > 0
mixed_numbers = type_by_num[has_supplier & has_callback].index.tolist()
print(f"혼재 번호: {len(mixed_numbers)}개")
print(mixed_numbers)

In [ ]:
# 혼재 번호 순회 (IDX 바꿔가며 확인)
IDX = 0  # ← 0, 1, 2 ...

if IDX < len(mixed_numbers):
    num = mixed_numbers[IDX]
    subset = df[df["receiver"] == num].sort_values("file_name")
    print(f"[{IDX+1}/{len(mixed_numbers)}] 번호: {num} | {len(subset)}건")
    print("=" * 80)
    for _, row in subset.iterrows():
        print(f"\n  파일: {row['file_name']}")
        print(f"  유형: {row['call_type']}  /  근거: {row.get('call_type_reason', '')}")
        print(f"  전사: {str(row.get('transcript', ''))[:400]}")
        print("  " + "-" * 58)
else:
    print("혼재 번호 없음")

In [ ]:
# 특정 번호 직접 검색
SEARCH = "01054598173"  # ← 번호 입력

subset = df[df["receiver"] == SEARCH].sort_values("file_name")
print(f"번호: {SEARCH} | {len(subset)}건")
print("=" * 80)
for _, row in subset.iterrows():
    print(f"\n  파일: {row['file_name']}")
    print(f"  유형: {row['call_type']}  /  근거: {row.get('call_type_reason', '')}")
    print(f"  요약: {row.get('summary', '')}")
    print(f"  전사: {str(row.get('transcript', ''))[:500]}")
    print("  " + "-" * 58)

---
## 2. "기타" 통화 내용 확인

In [ ]:
# 기타 전체 현황
etc_df = df[df["call_type"] == "기타"].copy()
print(f"기타 총 {len(etc_df)}건")

has_transcript = etc_df["transcript"].apply(lambda x: len(str(x).strip()) > 0)
print(f"  전사 있음: {has_transcript.sum()}건")
print(f"  전사 없음: {(~has_transcript).sum()}건")

print("\n[분류 근거 샘플]")
for reason in etc_df["call_type_reason"].dropna().unique()[:20]:
    print(f"  {reason}")

In [ ]:
# 기타 중 전사 있는 것 샘플 보기
SAMPLE_N = 10  # ← 볼 개수

etc_with_text = etc_df[etc_df["transcript"].apply(lambda x: len(str(x).strip()) > 10)]
sample = etc_with_text.sample(min(SAMPLE_N, len(etc_with_text)), random_state=42)

for _, row in sample.iterrows():
    print(f"\n파일: {row['file_name']}")
    print(f"근거: {row.get('call_type_reason', '')}")
    print(f"전사: {str(row.get('transcript', ''))[:400]}")
    print("-" * 70)

In [ ]:
# 기타 전체 전사 순서대로 보기 (OFFSET 바꿔가며 확인)
OFFSET = 0   # ← 시작 인덱스
PAGE = 5     # ← 한 번에 볼 개수

etc_with_text = etc_df[etc_df["transcript"].apply(lambda x: len(str(x).strip()) > 10)].reset_index(drop=True)
print(f"전사 있는 기타: {len(etc_with_text)}건 | 현재 {OFFSET}~{OFFSET+PAGE-1}")
print("=" * 70)

for _, row in etc_with_text.iloc[OFFSET:OFFSET + PAGE].iterrows():
    print(f"\n파일: {row['file_name']}")
    print(f"근거: {row.get('call_type_reason', '')}")
    print(f"전사: {str(row.get('transcript', ''))[:500]}")
    print("-" * 70)

---
## 3. "기타" 재분류 (Ollama 사용)
전사가 있는 기타 통화를 03과 동일한 분류 프롬프트로 다시 분류하고, 필요하면 체크포인트를 업데이트합니다.

In [ ]:
import re

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "qwen2.5:3b"
MAX_CHARS = 1800
VALID_TYPES = {"고객콜백", "공급업체", "기타"}

# 03_analyze_outbound_calls.ipynb와 동일한 언어 강제/재시도 로직 (qwen2.5의 중국어 혼입 방지)
LANG_GUARD = (
    "!!! IMPORTANT: You MUST respond ONLY in Korean (한국어). "
    "중국어(Chinese) 사용 절대 금지. 영어도 사용하지 마세요. "
    "모든 텍스트 필드 값은 반드시 한국어로만 작성하세요. !!!"
)
HAS_CHINESE_RE = re.compile(r"[一-鿿㐀-䶿]")


def has_chinese(text: str) -> bool:
    return bool(HAS_CHINESE_RE.search(text))


def build_type_prompt(transcript: str) -> str:
    return f"""{LANG_GUARD}

기프트코(판촉물 판매 사이트) 상담원이 외부로 건 발신 통화입니다.
반드시 한국어로만 답변하세요. Do not use Chinese or any other language.

【핵심 판단 기준: 상대방이 누구인가?】

상대방이 공급사·제조사·도매업체 직원이면 → "공급업체"
  · 기프트코에 상품을 납품하는 업체
  · 상담원이 재고·단가·납기·발주·인쇄조건 등을 확인하려 전화
  · 상대방이 "저희 상품", "저희 단가", "납기" 같은 공급자 표현 사용
  · 사이트 등록 단가·상품 정보 확인 (예: "단가 맞나요?", "등록 정보 확인")

상대방이 구매 고객·의뢰인이면 → "고객콜백"
  · 기프트코에서 상품을 사려는 일반 소비자 또는 구매 담당자
  · 상담원이 이전 문의에 답변하거나 주문·배송·결제를 처리하려 전화
  · 상대방이 "주문했는데", "언제 오나요" 같은 구매자 표현 사용

위 두 가지에 해당하지 않으면 → "기타"
  · 잘못 건 전화, 업무 무관, STT 불량, 판단 불가

규칙: JSON만 반환. 마크다운/코드블록 없이. 모든 텍스트 값은 반드시 한국어로 작성.

JSON:
{{"call_type":"고객콜백|공급업체|기타","reason":"상대방 역할 근거 한 줄"}}

전사문:
\"\"\"{transcript}\"\"\"
""".strip()


def safe_parse_json(text: str) -> dict:
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    s, e = text.find("{"), text.rfind("}")
    if s != -1 and e > s:
        try:
            return json.loads(text[s:e + 1])
        except Exception:
            pass
    return {}


def normalize_call_type(raw: str) -> str:
    raw = (raw or "").strip()
    if raw in VALID_TYPES:
        return raw
    if "고객" in raw:
        return "고객콜백"
    if "공급" in raw:
        return "공급업체"
    return "기타"


def reclassify(transcript: str, max_retries: int = 2) -> tuple[str, str]:
    prompt = build_type_prompt(transcript[:MAX_CHARS])
    payload = {"model": OLLAMA_MODEL, "prompt": prompt, "stream": False, "format": "json"}
    parsed, raw = {}, ""
    for attempt in range(max_retries + 1):
        resp = requests.post(OLLAMA_URL, json=payload, timeout=(30, 120))
        resp.raise_for_status()
        raw = resp.json().get("response", "")
        parsed = safe_parse_json(raw)
        if not has_chinese(raw):
            break
    return normalize_call_type(parsed.get("call_type", "")), parsed.get("reason", "")


# Ollama 확인
try:
    r = requests.get("http://localhost:11434/api/tags", timeout=5)
    print("Ollama OK:", [m["name"] for m in r.json().get("models", [])])
except Exception as e:
    print(f"Ollama 연결 실패: {e}")

In [ ]:
# 기타 중 전사 있는 것만 재분류
# ※ 전체 실행하면 시간이 걸립니다. TEST_ONLY=True 로 5건만 먼저 확인
TEST_ONLY = True   # ← False 로 바꾸면 전체 실행
TEST_COUNT = 5

etc_with_text = etc_df[
    etc_df["transcript"].apply(lambda x: len(str(x).strip()) > 10)
].copy()

targets = etc_with_text.head(TEST_COUNT) if TEST_ONLY else etc_with_text
print(f"재분류 대상: {len(targets)}건 (TEST_ONLY={TEST_ONLY})")
print("=" * 70)

reclassified = []
for idx, (_, row) in enumerate(targets.iterrows(), 1):
    transcript = str(row.get("transcript", ""))
    new_type, new_reason = reclassify(transcript)
    reclassified.append({
        "file_name": row["file_name"],
        "old_type": row["call_type"],
        "new_type": new_type,
        "new_reason": new_reason,
        "transcript": transcript[:200],
    })
    changed = "✓ 변경" if new_type != row["call_type"] else "  동일"
    print(f"[{idx:3d}] {changed} | {row['call_type']} → {new_type} | {new_reason[:50]}")

reclass_df = pd.DataFrame(reclassified)
print("\n재분류 결과:")
print(reclass_df["new_type"].value_counts())

In [ ]:
# 재분류 결과 체크포인트에 반영 (TEST_ONLY=False 로 전체 실행한 뒤 실행하세요)
# ※ 이 셀은 위 셀에서 TEST_ONLY=False 로 전체 실행한 뒤 실행하세요
APPLY = False  # ← True 로 바꾸면 체크포인트 업데이트

if APPLY:
    updated = {r["file_name"]: r for r in reclassified}
    changed_count = 0
    for result in ckpt["results"]:
        fname = result.get("file_name", "")
        if fname in updated and updated[fname]["new_type"] != "기타":
            result["call_type"] = updated[fname]["new_type"]
            result["call_type_reason"] = updated[fname]["new_reason"] + " [재분류]"
            changed_count += 1

    tmp = CHECKPOINT_FILE + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(ckpt, f, ensure_ascii=False, default=str)
    os.replace(tmp, CHECKPOINT_FILE)
    print(f"체크포인트 업데이트 완료: {changed_count}건 변경")
    print(f"저장: {CHECKPOINT_FILE}")
    print("주의: outbound_call_classification.csv/json에는 반영되지 않았습니다. "
          "03 노트북 또는 scripts/analyze_outbound_calls.py를 다시 실행해 재저장하세요.")
else:
    print("APPLY=False — 체크포인트 변경 없음")
    print("적용하려면 APPLY = True 로 변경 후 재실행")

---
## 4. 번호별 주요 분류 집계 (참고)

In [ ]:
dominant = df.groupby("receiver")["call_type"].agg(
    lambda x: x.value_counts().index[0]
).rename("dominant_type")
count_per_num = df.groupby("receiver").size().rename("call_count")
summary = pd.concat([dominant, count_per_num], axis=1).sort_values("call_count", ascending=False)
print("번호별 주요 분류 (통화 많은 순)")
summary.head(30)